In [206]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
import pickle

In [207]:
df = pd.read_csv('student.csv')
df.sample(5)

,Student_ID,Age,Gender,Academic_Year,Department,Sleep_Hours,Screen_Time,Attendance,Assignment_Load,Exam_Pressure,...,Heart_Rate,Stress_Score,Stress_Level,Anxiety_Score,Anxiety_Level,Mood_State,Previous_Intervention,Intervention_Response,Reward_Score,Final_State
441,442,23,Female,1st,ARTS,7.5,2.8,89.9,2,6,...,67,4.13,Medium,4.60,Medium,Neutral,Peer Support,0.67,6.50,Neutral
2711,2712,27,Male,1st,ARTS,8.8,6.7,66.4,2,8,...,79,4.69,Medium,4.18,Medium,Neutral,Peer Support,0.57,6.97,Neutral
8433,8434,18,Female,3rd,EEE,4.8,6.0,88.0,2,2,...,69,3.12,Low,4.31,Medium,Fatigued,No Action,0.39,5.10,Neutral
1893,1894,26,Female,2nd,ARTS,8.8,5.6,83.5,5,7,...,114,5.78,Medium,6.32,Medium,Neutral,Peer Support,0.53,4.54,Stress
4818,4819,20,Female,3rd,MECH,5.2,5.6,66.1,8,5,...,69,5.09,Medium,3.28,Low,Neutral,Motivational Message,0.61,7.01,Relaxed


In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             10000 non-null  int64  
 1   Age                    10000 non-null  int64  
 2   Gender                 10000 non-null  object 
 3   Academic_Year          10000 non-null  object 
 4   Department             10000 non-null  object 
 5   Sleep_Hours            10000 non-null  float64
 6   Screen_Time            10000 non-null  float64
 7   Attendance             10000 non-null  float64
 8   Assignment_Load        10000 non-null  int64  
 9   Exam_Pressure          10000 non-null  int64  
 10  Social_Support         10000 non-null  int64  
 11  Physical_Activity      10000 non-null  float64
 12  Caffeine_Intake        10000 non-null  int64  
 13  Study_Hours            10000 non-null  float64
 14  Facial_Emotion         10000 non-null  object 
 15  Hea

In [209]:
df.Academic_Year.value_counts()

Academic_Year
4th    2583
1st    2510
3rd    2471
2nd    2436
Name: count, dtype: int64

In [210]:
x = df.drop(['Student_ID', 'Final_State'], axis=1)
y = df['Final_State']

In [211]:
df.Final_State.value_counts()

Final_State
Neutral    7628
Relaxed    1190
Stress     1182
Name: count, dtype: int64

In [212]:
df.isnull().sum()

Student_ID               0
Age                      0
Gender                   0
Academic_Year            0
Department               0
Sleep_Hours              0
Screen_Time              0
Attendance               0
Assignment_Load          0
Exam_Pressure            0
Social_Support           0
Physical_Activity        0
Caffeine_Intake          0
Study_Hours              0
Facial_Emotion           0
Heart_Rate               0
Stress_Score             0
Stress_Level             0
Anxiety_Score            0
Anxiety_Level            0
Mood_State               0
Previous_Intervention    0
Intervention_Response    0
Reward_Score             0
Final_State              0
dtype: int64

In [213]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

x_train.shape, y_train.shape , x_test.shape, y_test.shape

((8000, 23), (8000,), (2000, 23), (2000,))

In [214]:
df.Anxiety_Level.shape , df.Academic_Year.shape , df.Stress_Level.shape

((10000,), (10000,), (10000,))

In [215]:
# encoding catagorical data -> ordinal encoding
ordinal_cols = ['Anxiety_Level', 'Stress_Level', 'Academic_Year']

oe = OrdinalEncoder(categories=[
    ['Low','Medium','High'],
    ['Low','Medium','High'],
    ['1st','2nd','3rd','4th']
])

x_train[ordinal_cols] = oe.fit_transform(x_train[ordinal_cols])

In [216]:
# encoding categorical data -> one hot encoding
ohe = OneHotEncoder(drop='first', sparse_output=False)

x_train_new = ohe.fit_transform(
    x_train[['Facial_Emotion','Department','Gender',
             'Previous_Intervention','Mood_State']]
)

x_test_new = ohe.transform(
    x_test[['Facial_Emotion','Department','Gender',
            'Previous_Intervention','Mood_State']]
)

In [217]:
# Label encoding the target variable
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [218]:
# Model

# dt = DecisionTreeClassifier(
#     max_depth=5,
#     random_state=42
# )
# dt.fit(x_train_new, y_train)
# y_pred = dt.predict(x_test_new)

from sklearn.ensemble import RandomForestClassifier

rr = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
rr.fit(x_train_new, y_train)
y_pred = rr.predict(x_test_new)

In [219]:
#Accuracy Score

accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

Accuracy: 0.75


In [220]:
train_acc = rr.score(x_train_new, y_train)
test_acc = rr.score(x_test_new, y_test)
print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

Training Accuracy: 0.766
Testing Accuracy: 0.75


In [221]:
import pickle

with open('stress_prediction_model.pkl', 'wb') as file:
    pickle.dump(rr, file)